# 4. Production Encoding with ColumnTransformer & Pipelines

This notebook demonstrates the end-to-end industry standard for categorical encoding:
1. Handling **multiple column types simultaneously** (Nominal, Ordinal, High-Cardinality, and Numerical).
2. Proper **Train/Test Splitting** to eliminate data leakage.
3. Fitting the transformation strictly on `X_train` and applying it to `X_test`.
4. Seamlessly chaining preprocessing with a Machine Learning model using `sklearn.pipeline.Pipeline`.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, TargetEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

# 1. Create a realistic enterprise dataset with mixed feature types
data = {
    'Age': [25, 34, 45, 22, 28, 52, 40, 29, 36, 48, 23, 41],
    'Annual_Income': [35000, 78000, 120000, 28000, 65000, 140000, 95000, 52000, 88000, 110000, 31000, 99000],
    'Device_Type': ['Android', 'iOS', 'Android', 'Windows', 'iOS', 'iOS', 'Android', 'Windows', 'iOS', 'Android', 'Android', 'iOS'], # Nominal
    'Education_Level': ['High School', 'Bachelors', 'PhD', 'High School', 'Masters', 'PhD', 'Bachelors', 'Masters', 'Bachelors', 'PhD', 'High School', 'Masters'], # Ordinal
    'City_Pincode': ['500001', '560001', '400001', '500001', '560001', '110001', '500001', '400001', '560001', '600001', '500001', '110001'], # High-Cardinality
    'Loan_Approved': [0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1] # Target (y)
}

df = pd.DataFrame(data)
print("=== 1. RAW MIXED DATASET ===")
display(df)
display(df.dtypes)

=== 1. RAW MIXED DATASET ===


,Age,Annual_Income,Device_Type,Education_Level,City_Pincode,Loan_Approved
0,25,35000,Android,High School,500001,0
1,34,78000,iOS,Bachelors,560001,1
2,45,120000,Android,PhD,400001,1
3,22,28000,Windows,High School,500001,0
4,28,65000,iOS,Masters,560001,1
5,52,140000,iOS,PhD,110001,1
6,40,95000,Android,Bachelors,500001,1
7,29,52000,Windows,Masters,400001,0
8,36,88000,iOS,Bachelors,560001,1
9,48,110000,Android,PhD,600001,1


Age                 int64
Annual_Income       int64
Device_Type        object
Education_Level    object
City_Pincode       object
Loan_Approved       int64
dtype: object

---
## Part 1: Train / Test Split (Preventing Data Leakage)

Before running any encoding or statistical transformations:
* We split data into training (`X_train`, `y_train`) and evaluation sets (`X_test`, `y_test`).
* Preprocessors must **never** see the test set during the `.fit()` stage.

In [2]:
# 1. Separate features (X) and target (y)
X = df.drop(columns=['Loan_Approved']).copy()
y = df['Loan_Approved'].copy()

# 2. Perform 75/25 Train-Test Split with reproducible random_state
X_train , X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

print(f"Training shape: {X_train.shape} | Testing shape: {X_test.shape}")

Training shape: (9, 5) | Testing shape: (3, 5)


---
## Part 2: Building the Multi-Encoder `ColumnTransformer`

We define transformation strategies based on feature semantics:
* **Nominal (`Device_Type`):** `OneHotEncoder(handle_unknown='ignore')`
* **Ordinal (`Education_Level`):** `OrdinalEncoder(categories=[education_order])`
* **High-Cardinality (`City_Pincode`):** `TargetEncoder(cv=3, smooth='auto')`
* **Numerical (`Age`, `Annual_Income`):** Passed through untouched via `remainder='passthrough'`

In [3]:
# Define explicit rank hierarchy for ordinal features
educational_order = ['High School', 'Bachelors', 'Masters', 'PhD']

# Build the preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('nominal_ohe', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), ['Device_Type']),
        ('ordinal_ohe', OrdinalEncoder(categories=[educational_order]), ['Education_Level']),
        ('target_enc', TargetEncoder(cv=3, smooth='auto', random_state=42), ['City_Pincode'])
    ],
    remainder='passthrough',
    verbose_feature_names_out=False 
).set_output(transform='pandas')

# Fit and transform training features
x_train_transformed = preprocessor.fit_transform(X_train, y_train)

# Transform test features strictly using learned train parameters
x_test_transformed = preprocessor.transform(X_test)

print("=== 2. TRANSFORMED TRAIN MATRIX ===")
display(x_train_transformed)
print("\n=== 3. TRANSFORMED TEST MATRIX ===")
display(x_test_transformed)

=== 2. TRANSFORMED TRAIN MATRIX ===


,Device_Type_Android,Device_Type_Windows,Device_Type_iOS,Education_Level,City_Pincode,Age,Annual_Income
3,0.0,1.0,0.0,0.0,0.000000,22,28000
5,0.0,0.0,1.0,3.0,1.000000,52,140000
11,0.0,0.0,1.0,2.0,1.000000,41,99000
10,1.0,0.0,0.0,0.0,0.000000,23,31000
9,1.0,0.0,0.0,3.0,0.666667,48,110000
0,1.0,0.0,0.0,0.0,0.000000,25,35000
8,0.0,0.0,1.0,1.0,0.666667,36,88000
1,0.0,0.0,1.0,1.0,0.666667,34,78000
2,1.0,0.0,0.0,3.0,0.666667,45,120000



=== 3. TRANSFORMED TEST MATRIX ===


,Device_Type_Android,Device_Type_Windows,Device_Type_iOS,Education_Level,City_Pincode,Age,Annual_Income
7,0.0,1.0,0.0,2.0,1.0,29,52000
4,0.0,0.0,1.0,2.0,1.0,28,65000
6,1.0,0.0,0.0,1.0,0.0,40,95000


---
## Part 3: Wrapping Preprocessor & Model in a Single `Pipeline`

In production, you package the preprocessor and the ML model into an end-to-end `Pipeline`.
* When raw live records arrive at an API, the pipeline automatically applies all encodings and outputs predictions with a single call to `.predict()`.

In [4]:
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=50, random_state=42))
])

full_pipeline.fit(X_train, y_train)

y_pred = full_pipeline.predict(X_test)

print("=== 4. MODEL EVALUATION ON TEST DATA ===")
print(classification_report(y_test, y_pred))

=== 4. MODEL EVALUATION ON TEST DATA ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         1
           1       1.00      1.00      1.00         2

    accuracy                           1.00         3
   macro avg       1.00      1.00      1.00         3
weighted avg       1.00      1.00      1.00         3



---
## Part 4: Key Pipeline Architecture Rules

1. **Split First:** Always execute `train_test_split` prior to any feature transformation.
2. **Fit on Train Only:** Call `.fit_transform()` strictly on `X_train` (and `y_train` if using TargetEncoder).
3. **Transform on Test:** Call `.transform()` on `X_test` to prevent data leakage and schema mismatch.
4. **Single Artifact Deployment:** Saving `full_pipeline` (e.g., via `joblib`) ensures your inference API takes raw JSON input and returns predictions seamlessly.